# 05 – Orchestration agentique avec LangChain
**Travail demandé 6** : l'agent choisit lui-même ses outils :

| Outil | Rôle |
|---|---|
| `rechercher_code_defaut` | recherche exacte d'un code (2310, A2B1, 99.13…) |
| `rechercher_texte` | procédures, explications, consignes |
| `rechercher_tableaux` | valeurs, paramètres, intervalles |
| `rechercher_images` | schémas et figures (CLIP + descriptions) |

`create_agent` (LangChain ≥ 1.0) gère la boucle : raisonnement → appel d'outil → observation → réponse.

In [4]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from src import config
PDF = config.PDF_DIR / "ABB_ACS580_Guide_Variateur.pdf"
print("Projet :", ROOT, "| PDF existe :", PDF.exists())

Projet : c:\Users\OMEN\Downloads\parti 1 projet agent | PDF existe : True


In [5]:
from src.agent import MaintenanceAgent
from IPython.display import Markdown, display
agent = MaintenanceAgent('pgvector')
def run(q, history=None):
    res = agent.ask(q, history=history)
    for i, s in enumerate(res['steps'], 1):
        print(f"{i}. {s['tool']}({s['args']})")
    display(Markdown(res['answer']))
    return res

ConnectionTimeout: connection timeout expired
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: '5433', hostaddr: '::1': connection timeout expired
- host: 'localhost', port: '5433', hostaddr: '127.0.0.1': connection timeout expired

In [ ]:
r1 = run('Le variateur affiche 2310, que dois-je faire ?')

NameError: name 'run' is not defined

In [ ]:
r2 = run('Quelle est la différence entre les défauts 2330 et 3181 ?')

In [ ]:
r3 = run('Montre-moi le logigramme de dépannage de la surintensité')
[(i['page'], i['image_path']) for i in r3['images']]

## Conversation avec mémoire (historique)

In [ ]:
h = [{'role': 'user', 'content': 'Que signifie le défaut 5091 ?'}]
first = agent.ask(h[0]['content'])
h.append({'role': 'assistant', 'content': first['answer']})
r4 = run('Et si seul FA81 apparaît en plus, quelle est la cause probable ?', history=h)

## Comparaison RAG simple vs agent
L'agent appelle la recherche exacte pour les codes, ce qui évite les confusions entre codes proches (2310 / 2330 / 2340).